# Homework 1 submission

> Cohort '26, ML zoomcamp.

## Environment setup

For the homework, I'm managing the environment and packages with `uv`. 

### Thoughts

#### Ana(conda)

When first starting out in data science, I worked with `conda`, since it was the gold standard for data science and machine learning workflows. Being completely fresh to computer science, I downloaded the Anaconda distribution, which I learn now to be rather bloated. I then moved `miniconda`, which is the minimal installer that provides us with the `conda` package manager and lets us install only what we need for a given project. However, in some cases, I found that the dependency resolution can take some time, which makes sense given that it is trying to solve for a much larger space of non-Python packages as well. 

#### Poetry and `uv`
This slowness I experienced with `conda` brought me to Poetry and `uv`, which are specific to the Python ecosystem. Leaving out the optimisation mechanics (because I don't really understand them anyway), `uv` is written in Rust, and Poetry in Python. With the clear language advantage, `uv` operations run natively at system speed without any interpreter execution overhead. And a few RTFM sessions later, I settled with `uv` whenever managing Python-involved projects.

## Question 1. Pandas version

The `pandas` version installed in the current environment is `3.0.5`.

In [1]:
import pandas as pd
print(pd.__version__)

3.0.5


## Getting the data

Here, I'm parsing the file directly from the web URL into a `pd.DataFrame` primarily for convenience and quick prototyping.

### Note:

While loading data via URL works well for prototyping, reading from a local disk or a dedicated database is better for production pipelines. Local disk I/O is significantly faster (more **performant**) than fetching data over a network, which is constrained by bandwidth and latency. Additionally, it's more **reliable**. Local files are not bound by network constraints, i.e., they don't fail during network drops or server downtimes. Finally, it helps with pipeline **stability**. Remote datasets can be modified, relocated, or deleted without warning. Hardcoding URLs introduces a point of failure that will break the script written below.

In [2]:
URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(URL)

## Question 2. Records count

There are 10,000 rows in the dataset.

In [3]:
print(len(df))

10000


## Question 3: Fuel types

There are three fuel types: Gasoline, Diesel, and Hybrid.

### Note:

The `<ArrowStringArray>` output is due to pandas running the `PyArrow` backend, which I have installed for reading and writing parquet files.

In [4]:
print(len(df.fuel_type.unique()))
print(df.fuel_type.unique())

3
<ArrowStringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str


## Question 4: Missing values

There are 2 columns, namely `horsepower`, and `acceleration` with missing values.

In [5]:
print((df.isna().sum() > 0).sum())
print(df.columns[(df.isna().sum() > 0)])

2
Index(['horsepower', 'acceleration'], dtype='str')


## Question 5. Max fuel efficiency

Process:

1. Quick look at the columns first to find the column that might store continents (`origin`)
2. Filter on "Asia" values
3. Run .max() on `fuel_efficiency_mpg`

The maximum fuel efficiency of Asian cars is 41.2.

In [6]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [7]:
print(df[df.origin == "Asia"].fuel_efficiency_mpg.max())

41.2


## Question 6. Median value of horsepower

The median shifts lower upwards. Since we've replaced 877 observations with the modal value (which is lower than the median), the median is pulled down, naturally.

In [8]:
print(df.horsepower.median())

254.0


In [9]:
print(df.horsepower.mode()[0])

252.0


In [10]:
df.horsepower.isna().sum()

np.int64(877)

In [11]:
mode_hp = df.horsepower.mode()[0]
df_imp = df.copy()
df_imp.horsepower = df_imp.horsepower.fillna(mode_hp)

In [12]:
print(df_imp.horsepower.median())

252.0


## Question 7. Sum of weights

The weights are computed via the following (normal equation) formula:

$$w = (X^T X)^{-1}(X^Ty)$$

The weights vector `w` represent the coefficient estimates of linear regression. In other words, the slopes or parameters of the model. We can conclude that as the weight of the vehicle and the model year increase by one relative unit, the fuel efficiency increases by `0.136` and `0.233` units (engine displacement, presumably), respectively.

In [13]:
import numpy as np

df_asia = df_imp[df_imp.origin == "Asia"]
df_asia_two_cols = df_asia[["vehicle_weight", "model_year"]]
X = df_asia_two_cols.head(7).to_numpy()

**Computing the first half (Inverse Gram Matrix):**

This matrix captures the variance and covariance between the selected features. This is also why when two or more variables are multicollinear, it causes the matrix to collapse to (or near) zero, the inverse of which then causes the variance of the coefficients to explode, making the model parameters wildly unstable.

In [14]:
XTX = X.T.dot(X)

In [15]:
XTX_inv = np.linalg.inv(XTX)

**Computing the second half (Cross-Product Vector):** 

This measures the mathematical relationship between target `y` and the feature matrix `X`.

In [16]:
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
w = (XTX_inv @ X.T) @ y

In [17]:
print("Coefficient estimates:")
print(f"vehicle_weight: {w[0]}")
print(f"model_year: {w[1]}\n")
print(f"Sum of weights: {sum(w)}")

Coefficient estimates:
vehicle_weight: 0.13644776561569216
model_year: 0.23274920343358296

Sum of weights: 0.3691969690492751
